# 📖 Notebook 2: Conflict Resolution with CRDTs

In the previous notebook we learned about OT, where a **central server** transforms operations to keep everyone in sync. CRDTs (Conflict-free Replicated Data Types) take a completely different approach: they make operations **commutative** — meaning they can be applied in **any order** and still produce the same result.

## Learning Objectives

By the end of this notebook, you'll understand:
- What CRDTs are and why they exist
- How position-based IDs make inserts order-independent
- How tombstones handle deletions
- OT vs CRDT trade-offs for real systems

## 🛠️ Setup

```bash
cd system-designs/google-docs
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

## 🤔 Why CRDTs?

OT requires a **central server** — every operation must pass through it so the server can enforce ordering. But what if:

- You want **offline editing** (like Apple Notes)?
- You want **peer-to-peer** collaboration (no server)?
- You want to **scale beyond one server** per document?

CRDTs solve this by guaranteeing that **no matter what order operations arrive, every client converges on the same document**.

```
OT:    "Apply operations in the right order"     → needs a server
CRDTs: "Make operations work in ANY order"        → no server needed
```

## 🔑 The Two Key Tricks of Text CRDTs

### Trick 1: Unique Position IDs

Instead of using integer positions (like `INSERT(5, "x")`), CRDTs give every character a **unique ID** that can be infinitely subdivided — like decimal numbers between 0 and 1.

```
Traditional (OT):     H  e  l  l  o  !
                      0  1  2  3  4  5    ← integer positions (shift when you insert!)

CRDT:                 H    e    l    l    o    !
                     0.1  0.2  0.3  0.4  0.5  0.6   ← unique IDs (never change!)
```

To insert between `o` (0.5) and `!` (0.6), just pick a number between them: 0.55.

### Trick 2: Tombstones for Deletion

When you delete a character, you don't actually remove it. Instead, you **mark it as deleted** (a "tombstone"). The character is hidden from view but stays in the data structure to maintain position references.

```
Before delete:  H  e  l  l  o  !     (all visible)
After delete !: H  e  l  l  o  [!]   (! is tombstoned, hidden from display)
```

In [1]:
import random
import time


class CRDTChar:
    """A single character in our CRDT document.
    
    Each character has:
    - position_id: a unique float that determines ordering
    - char: the actual character
    - deleted: whether this character has been tombstoned
    - site_id: which user created this character (for tie-breaking)
    """
    def __init__(self, position_id, char, site_id, deleted=False):
        self.position_id = position_id
        self.char = char
        self.site_id = site_id
        self.deleted = deleted
    
    def __repr__(self):
        status = "🪦" if self.deleted else "✓"
        return f"{status} '{self.char}' @{self.position_id:.4f} (site={self.site_id})"


class CRDTDocument:
    """A simple CRDT-based collaborative text document.
    
    This is a simplified version of algorithms like LSEQ or Logoot.
    Real implementations use variable-length IDs, but floats are
    easier to understand.
    """
    
    def __init__(self, site_id):
        self.site_id = site_id
        self.chars = []  # sorted list of CRDTChar
    
    def _find_position(self, position_id):
        """Find where to insert a char to keep the list sorted by position_id."""
        for i, c in enumerate(self.chars):
            if c.position_id > position_id:
                return i
            if c.position_id == position_id and c.site_id > self.site_id:
                return i  # tie-break by site_id
        return len(self.chars)
    
    def _generate_position_id(self, index):
        """Generate a unique position ID between two adjacent characters."""
        # Get the position IDs of neighbors
        if index == 0:
            before = 0.0
        else:
            # Find the visible char before this index
            visible_count = 0
            before = 0.0
            for c in self.chars:
                if not c.deleted:
                    if visible_count == index - 1:
                        before = c.position_id
                        break
                    visible_count += 1
        
        # Find the visible char at this index (or end)
        visible_count = 0
        after = 1.0
        for c in self.chars:
            if not c.deleted:
                if visible_count == index:
                    after = c.position_id
                    break
                visible_count += 1
        
        # Pick a random point between before and after
        return before + (after - before) * random.uniform(0.3, 0.7)
    
    def insert(self, index, char):
        """Insert a character at visible index. Returns the operation."""
        pos_id = self._generate_position_id(index)
        crdt_char = CRDTChar(pos_id, char, self.site_id)
        insert_pos = self._find_position(pos_id)
        self.chars.insert(insert_pos, crdt_char)
        return {"type": "insert", "position_id": pos_id, "char": char, "site_id": self.site_id}
    
    def delete(self, index):
        """Delete the character at visible index. Returns the operation."""
        visible_count = 0
        for c in self.chars:
            if not c.deleted:
                if visible_count == index:
                    c.deleted = True
                    return {"type": "delete", "position_id": c.position_id, "site_id": self.site_id}
                visible_count += 1
        return None
    
    def apply_remote(self, op):
        """Apply a remote operation (from another site)."""
        if op["type"] == "insert":
            crdt_char = CRDTChar(op["position_id"], op["char"], op["site_id"])
            insert_pos = self._find_position(op["position_id"])
            self.chars.insert(insert_pos, crdt_char)
        elif op["type"] == "delete":
            for c in self.chars:
                if c.position_id == op["position_id"]:
                    c.deleted = True
                    break
    
    def get_text(self):
        """Get the visible text (excluding tombstoned characters)."""
        return "".join(c.char for c in self.chars if not c.deleted)
    
    def get_all_chars(self):
        """Get all characters including tombstones (for debugging)."""
        return self.chars


print("CRDT Document class defined! ✅")

CRDT Document class defined! ✅


In [2]:
# Demo: Basic CRDT operations
random.seed(42)  # for reproducible positions

doc = CRDTDocument(site_id="alice")

# Build "Hello!" one character at a time
for i, ch in enumerate("Hello!"):
    doc.insert(i, ch)

print(f"Document text: '{doc.get_text()}'")
print(f"\nInternal representation (with position IDs):")
for c in doc.get_all_chars():
    print(f"  {c}")

print(f"\n💡 Notice each character has a unique position_id.")
print(f"   These IDs NEVER change, even when other chars are inserted.")

Document text: 'Hello!'

Internal representation (with position IDs):
  ✓ 'H' @0.5558 (site=alice)
  ✓ 'e' @0.6935 (site=alice)
  ✓ 'l' @0.8192 (site=alice)
  ✓ 'l' @0.8896 (site=alice)
  ✓ 'o' @0.9552 (site=alice)
  ✓ '!' @0.9808 (site=alice)

💡 Notice each character has a unique position_id.
   These IDs NEVER change, even when other chars are inserted.


In [3]:
# Demo: Tombstones — deleting keeps the character but marks it invisible
random.seed(42)

doc = CRDTDocument(site_id="alice")
for i, ch in enumerate("Hello!"):
    doc.insert(i, ch)

print(f"Before delete: '{doc.get_text()}'")
print(f"Chars in memory: {len(doc.get_all_chars())}")
print()

# Delete the '!' (visible index 5)
op = doc.delete(5)
print(f"After deleting '!': '{doc.get_text()}'")
print(f"Chars in memory: {len(doc.get_all_chars())}  ← still 6!")
print()

print("Internal state:")
for c in doc.get_all_chars():
    print(f"  {c}")

print(f"\n💡 The '!' is still in memory (tombstoned 🪦) but hidden from display.")
print(f"   This is the trade-off: CRDTs use more memory because documents only grow.")

Before delete: 'Hello!'
Chars in memory: 6

After deleting '!': 'Hello'
Chars in memory: 6  ← still 6!

Internal state:
  ✓ 'H' @0.5558 (site=alice)
  ✓ 'e' @0.6935 (site=alice)
  ✓ 'l' @0.8192 (site=alice)
  ✓ 'l' @0.8896 (site=alice)
  ✓ 'o' @0.9552 (site=alice)
  🪦 '!' @0.9808 (site=alice)

💡 The '!' is still in memory (tombstoned 🪦) but hidden from display.
   This is the trade-off: CRDTs use more memory because documents only grow.


## 🔄 Concurrent Editing with CRDTs

The magic of CRDTs: two users can edit simultaneously, receive each other's operations in **any order**, and still converge on the same document.

In [4]:
# Two users editing the same document concurrently
random.seed(100)

# Both start with the same document
alice_doc = CRDTDocument(site_id="alice")
bob_doc = CRDTDocument(site_id="bob")

# Initialize both with "Hello"
init_ops = []
for i, ch in enumerate("Hello"):
    op = alice_doc.insert(i, ch)
    init_ops.append(op)

# Sync initial state to Bob
for op in init_ops:
    bob_doc.apply_remote(op)

print(f"Alice's document: '{alice_doc.get_text()}'")
print(f"Bob's document:   '{bob_doc.get_text()}'")
print()

# Now both edit CONCURRENTLY (without seeing each other's changes)
# Alice inserts ", world" at position 5
alice_ops = []
for i, ch in enumerate(", world"):
    op = alice_doc.insert(5 + i, ch)
    alice_ops.append(op)

# Bob inserts "!" at position 5
bob_ops = []
op = bob_doc.insert(5, "!")
bob_ops.append(op)

print(f"Alice (before sync): '{alice_doc.get_text()}'")
print(f"Bob (before sync):   '{bob_doc.get_text()}'")
print()

# Now sync: Alice receives Bob's ops, Bob receives Alice's ops
for op in bob_ops:
    alice_doc.apply_remote(op)
for op in alice_ops:
    bob_doc.apply_remote(op)

print(f"Alice (after sync):  '{alice_doc.get_text()}'")
print(f"Bob (after sync):    '{bob_doc.get_text()}'")
print()

converged = alice_doc.get_text() == bob_doc.get_text()
print(f"✅ Documents converged: {converged}")
print(f"\n💡 No central server needed! Both clients applied operations")
print(f"   independently and arrived at the same result.")

Alice's document: 'Hello'
Bob's document:   'Hello'

Alice (before sync): 'Hello, world'
Bob (before sync):   'Hello!'

Alice (after sync):  'Hello,! world'
Bob (after sync):    'Hello,! world'

✅ Documents converged: True

💡 No central server needed! Both clients applied operations
   independently and arrived at the same result.


In [5]:
# Let's prove order doesn't matter: apply ops in DIFFERENT orders
random.seed(200)

# Create three independent documents
doc_a = CRDTDocument(site_id="alice")
doc_b = CRDTDocument(site_id="bob")
doc_c = CRDTDocument(site_id="charlie")

# Initialize all with "ABC"
init_ops = []
for i, ch in enumerate("ABC"):
    op = doc_a.insert(i, ch)
    init_ops.append(op)
for op in init_ops:
    doc_b.apply_remote(op)
    doc_c.apply_remote(op)

# Three concurrent edits:
op1 = doc_a.insert(1, "x")   # Alice inserts 'x' after 'A'
op2 = doc_b.insert(2, "y")   # Bob inserts 'y' after 'B'
op3 = doc_c.insert(3, "z")   # Charlie inserts 'z' after 'C'

# Apply in different orders to each document
# Doc A: already has op1, apply op2 then op3
doc_a.apply_remote(op2)
doc_a.apply_remote(op3)

# Doc B: already has op2, apply op3 then op1 (DIFFERENT ORDER)
doc_b.apply_remote(op3)
doc_b.apply_remote(op1)

# Doc C: already has op3, apply op1 then op2 (DIFFERENT ORDER)
doc_c.apply_remote(op1)
doc_c.apply_remote(op2)

print("Three users, three different operation orders:")
print(f"  Alice   (op1 → op2 → op3): '{doc_a.get_text()}'")
print(f"  Bob     (op2 → op3 → op1): '{doc_b.get_text()}'")
print(f"  Charlie (op3 → op1 → op2): '{doc_c.get_text()}'")
print()

all_same = doc_a.get_text() == doc_b.get_text() == doc_c.get_text()
print(f"✅ All documents identical: {all_same}")
print(f"\n💡 This is the CRDT guarantee: convergence regardless of operation order!")

Three users, three different operation orders:
  Alice   (op1 → op2 → op3): 'AxByCz'
  Bob     (op2 → op3 → op1): 'AxByCz'
  Charlie (op3 → op1 → op2): 'AxByCz'

✅ All documents identical: True

💡 This is the CRDT guarantee: convergence regardless of operation order!


## 📊 OT vs CRDTs: A Comparison

Let's compare the two approaches side by side:

In [6]:
# Let's measure the memory difference
import sys
random.seed(42)

# Create a CRDT document, write 100 chars, then delete 50
crdt_doc = CRDTDocument(site_id="alice")
text = "The quick brown fox jumps over the lazy dog. " * 2  # ~90 chars
for i, ch in enumerate(text):
    crdt_doc.insert(i, ch)

visible_before = len(crdt_doc.get_text())
total_before = len(crdt_doc.get_all_chars())

# Delete every other character (simulate heavy editing)
delete_count = 0
for i in range(0, len(text), 2):
    idx = i - delete_count  # adjust for characters already removed from visible
    if idx < len(crdt_doc.get_text()):
        crdt_doc.delete(idx)
        delete_count += 1

visible_after = len(crdt_doc.get_text())
total_after = len(crdt_doc.get_all_chars())

print("📊 CRDT Memory Usage")
print("=" * 50)
print(f"Before deletions:")
print(f"  Visible chars: {visible_before}")
print(f"  Total in memory: {total_before}")
print()
print(f"After deleting {delete_count} characters:")
print(f"  Visible chars: {visible_after}")
print(f"  Total in memory: {total_after}  ← same! Tombstones still there")
print(f"  Memory overhead: {total_after - visible_after} tombstoned chars ({((total_after - visible_after) / total_after * 100):.0f}%)")
print()
print("💡 In OT, deleted text is just gone. In CRDTs, it stays as tombstones.")
print("   For long-lived documents with heavy editing, this adds up.")

📊 CRDT Memory Usage
Before deletions:
  Visible chars: 90
  Total in memory: 90

After deleting 45 characters:
  Visible chars: 45
  Total in memory: 90  ← same! Tombstones still there
  Memory overhead: 45 tombstoned chars (50%)

💡 In OT, deleted text is just gone. In CRDTs, it stays as tombstones.
   For long-lived documents with heavy editing, this adds up.


In [7]:
# Summary comparison table

print("📊 OT vs CRDTs: When to Use Each")
print("=" * 70)
print()
print(f"{'Feature':<30} {'OT':<20} {'CRDTs':<20}")
print("-" * 70)

comparisons = [
    ("Central server",          "Required ⚠️",     "Not needed ✅"),
    ("Offline support",         "Limited ⚠️",      "Excellent ✅"),
    ("Memory usage",            "Low ✅",           "Higher (tombstones) ⚠️"),
    ("Implementation",          "Complex ⚠️",      "Complex ⚠️"),
    ("Peer-to-peer",            "No ❌",            "Yes ✅"),
    ("Scaling",                 "1 server/doc ⚠️", "Unlimited ✅"),
    ("Conflict quality",        "Good ✅",          "Can be awkward ⚠️"),
    ("Text editing fit",        "Excellent ✅",     "Good ✅"),
]

for feature, ot, crdt in comparisons:
    print(f"{feature:<30} {ot:<20} {crdt:<20}")

print()
print("Real-world choices:")
print("  • Google Docs → OT (central server, low memory, proven)")
print("  • Figma → CRDT-inspired (design ops, custom implementation)")
print("  • Apple Notes → CRDT (offline-first across iCloud devices)")
print("  • Yjs (open source) → CRDT (great for building your own)")

📊 OT vs CRDTs: When to Use Each

Feature                        OT                   CRDTs               
----------------------------------------------------------------------
Central server                 Required ⚠️          Not needed ✅        
Offline support                Limited ⚠️           Excellent ✅         
Memory usage                   Low ✅                Higher (tombstones) ⚠️
Implementation                 Complex ⚠️           Complex ⚠️          
Peer-to-peer                   No ❌                 Yes ✅               
Scaling                        1 server/doc ⚠️      Unlimited ✅         
Conflict quality               Good ✅               Can be awkward ⚠️   
Text editing fit               Excellent ✅          Good ✅              

Real-world choices:
  • Google Docs → OT (central server, low memory, proven)
  • Figma → CRDT-inspired (design ops, custom implementation)
  • Apple Notes → CRDT (offline-first across iCloud devices)
  • Yjs (open source) → CRDT (great

## 🧹 Cleanup

In [8]:
print("🧹 No cleanup needed — all CRDT operations were in-memory.")

🧹 No cleanup needed — all CRDT operations were in-memory.


## 📚 Summary

### Key Takeaways

1. **CRDTs make operations commutative** — apply in any order, get the same result
2. **Unique position IDs** — each character gets an ID that never changes, even when neighbors are inserted
3. **Tombstones for deletion** — deleted characters stay in memory (trade-off: memory grows)
4. **No central server** — perfect for offline editing, peer-to-peer, and scaling
5. **Trade-off**: more memory, potentially awkward merge results at the same position

### For System Design Interviews

- Default to **OT** for Google Docs-style problems (what they actually use)
- Mention **CRDTs** as an alternative if the interviewer asks about offline mode or P2P
- Know the trade-offs: OT = low memory + central server; CRDTs = more memory + no server

### Next Up

In **Notebook 3**, we'll build the **real-time collaboration experience** using WebSockets — connecting to our doc server, sending live edits, and seeing other users' cursors.